# Fig. 8 and Table III: WSR versus runtime

Run cells in order from a fresh kernel. Settings are in `configs/experiments.json`; artifacts use the paths described in the README. Timings depend on hardware and software.


In [ ]:
from fthbf.runtime import (DATA_DIR, CHECKPOINT_DIR, checkpoint_path, load_artifact,
                           result_dir, read_config, seed_everything, record_run)
import torch
cfg = read_config("experiments.json")["Fig03"]
seed_everything(cfg["seed"])
record_run("Fig03", cfg)


In [ ]:
with torch.no_grad():
    # =====================================================
    # This script evaluates Runtime vs Optimized Sum-rate
    # Classical WMMSE + Unfolded WMMSE (Iter = 2,4,6) + other benchmarks
    # =====================================================

    import os
    import time
    import torch
    import numpy as np
    import matplotlib.pyplot as plt
    from tqdm import tqdm

    from solutions.trihybrid_WMMSE import TriHybridWMMSE, GNN
    from solutions.utils import computeWSR, channelDataset, HybridPrecodingBFGS, checkHybridConstraints
    from benchmark_algorithms.AOMO_alg import AOMOBeamformer
    from benchmark_algorithms.Decomp_alg import DecompBeamformer

    # =====================================================
    # Global settings
    # =====================================================
    cdtype = torch.cfloat
    NUM_SAMPLES = cfg["samples"]
    Nrf = cfg["rf_chains"]
    UWMMSE_I_LIST = cfg["unfolded_layers"]
    RESULT_DIR = result_dir("results_WSR_vs_RunTime")
    os.makedirs(RESULT_DIR, exist_ok=True)
    SEED = cfg["seed"]
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    # =====================================================
    # Load data
    # =====================================================
    SAVE_DIR = DATA_DIR
    data = load_artifact(os.path.join(SAVE_DIR, "probInsts_test.pt"))
    parameters = load_artifact(os.path.join(SAVE_DIR, "parameters.pt"))
    ProbInsts = data["ProbInsts"]
    Nt = parameters["Nt"]
    Nc = parameters["Nc"]
    P = parameters["P"]
    Mk = parameters["Mk"]
    eta = parameters["eta"]
    Ns_k = parameters["Ns_k"]
    sigma2 = parameters["sigma2"]
    dataset = channelDataset(ProbInsts, parameters)

    # =====================================================
    # Build & load UWMMSE models (ONCE)
    # =====================================================
    def load_uwmmse_models(I_max):
        in_dim = 2 * ProbInsts[0]["H"][0].numel() + 1
        GNNs_gamma = torch.nn.ModuleList()
        GNNs_u = torch.nn.ModuleList()
        for _ in range(I_max):
            GNNs_gamma.append(
                GNN(in_dim=in_dim,
                    hidden_dim=2 * Ns_k**2 + 4,
                    out_dim=2 * Ns_k**2,
                    num_layers=3,
                    dp_rate=0.1))
            GNNs_u.append(
                GNN(in_dim=in_dim,
                    hidden_dim=Nt + 10,
                    out_dim=Nt,
                    num_layers=3,
                    dp_rate=0.1))
        ckpt = load_artifact(checkpoint_path(I_max), map_location="cpu")

        for i in range(I_max):
            GNNs_gamma[i].load_state_dict(ckpt["GNNs_gamma_state_dict"][i])
            GNNs_u[i].load_state_dict(ckpt["GNNs_u_state_dict"][i])
        for m in list(GNNs_gamma) + list(GNNs_u):
            m.eval()

        print(f"[Info] Loaded UWMMSE model (I_max={I_max})")
        return GNNs_gamma, GNNs_u

    UWMMSE_MODELS = {I: load_uwmmse_models(I) for I in UWMMSE_I_LIST}

    # =====================================================
    # Algorithm implementations
    # =====================================================
    def run_classical_WMMSE(prob_inst):
        H_list = prob_inst["H"]
        K = prob_inst["K"]
        Ns = Ns_k * K

        u0 = [torch.ones(Nc, 1, dtype=cdtype)] * Nt
        v0 = [torch.ones(Ns, 1, dtype=cdtype)] * Nt
        for n in range(Nt):
            if torch.linalg.norm(v0[n]) > torch.sqrt(P):
                v0[n] = v0[n] / torch.linalg.norm(v0[n]) * torch.sqrt(P)

        t0 = time.time()
        # ------
        solver = TriHybridWMMSE(
            H_list=H_list,
            Ns_k=[Ns_k] * K,
            Mk=[Mk] * K,
            sigma2=[sigma2] * K,
            beta=[1.0] * K,
            P=[P] * Nt,
            eta=eta,
            u_init=u0,
            v_init=v0,
        )

        _, FA_opt, FD_opt, _, _ = solver.run_classical(I_max=200, verbose=False, tol=1e-6)
        FRF, FBB = HybridPrecodingBFGS(FD_opt, Nrf).solve(max_iter=100, use_B0=True, verbose=False)
        _, papc_ok, info = checkHybridConstraints(FRF, FBB, P, tol=1e-6, verbose=False)
        if not papc_ok:
            FBB = FBB / torch.sqrt(torch.tensor(info["max_papc_ratio"]))
        # ------
        runtime = time.time() - t0

        R, _ = computeWSR(FA_opt, FRF, FBB, H_list, parameters)
        return runtime, R.item()


    def run_UWMMSE(inst_idx, I_max):
        prob_inst = dataset[inst_idx]
        H_list = prob_inst["H_list"]
        K = prob_inst["K"]
        Ns = Ns_k * K

        u0 = [torch.ones(Nc, 1, dtype=cdtype)] * Nt
        v0 = [torch.ones(Ns, 1, dtype=cdtype)] * Nt
        for n in range(Nt):
            if torch.linalg.norm(v0[n]) > torch.sqrt(P):
                v0[n] = v0[n] / torch.linalg.norm(v0[n]) * torch.sqrt(P)

        node_features = prob_inst["node_feature"]
        edge_index = prob_inst["edge_index"]
        graph_id = torch.zeros(K, dtype=torch.int32)
        num_graphs = torch.tensor(1, dtype=torch.int32)

        GNNs_gamma, GNNs_u = UWMMSE_MODELS[I_max]

        delta_gamma_list, delta_u_list = [], []
        for i in range(I_max):
            delta_gamma_list.append(GNNs_gamma[i](node_features, edge_index))
            delta_u = GNNs_u[i](node_features, edge_index, graph_id, num_graphs)
            delta_u_list.append(delta_u.reshape(Nt, 1))

        t0 = time.time()
        # ------
        solver = TriHybridWMMSE(
            H_list=H_list,
            Ns_k=[Ns_k] * K,
            Mk=[Mk] * K,
            sigma2=[sigma2] * K,
            beta=[1.0] * K,
            P=[P] * Nt,
            eta=eta,
            u_init=u0,
            v_init=v0,
        )

        _, FA_opt, FD_opt, _, _ = solver.run_unfolded(dgamma_list=delta_gamma_list, du_list=delta_u_list, I_max=I_max, verbose=False)
        FRF, FBB = HybridPrecodingBFGS(FD_opt, Nrf).solve(max_iter=100, use_B0=True, verbose=False)
        _, papc_ok, info = checkHybridConstraints(FRF, FBB, P, tol=1e-6, verbose=False)
        if not papc_ok:
            FBB = FBB / torch.sqrt(torch.tensor(info["max_papc_ratio"]))
        # ------
        runtime = time.time() - t0

        R, _ = computeWSR(FA_opt, FRF, FBB, H_list, parameters)
        return runtime, R.item()

    def run_AOMO_wrapper(prob_inst):
        H_list = prob_inst["H"]
        K = prob_inst["K"]
        # convert to numpy for AOMO implementation
        H_list_np = [x.numpy() for x in H_list]    
        eta_np = [x.numpy().squeeze() for x in eta]

        t0 = time.time()
        # ------
        solver = AOMOBeamformer(
            H=H_list_np,
            Ns_k=[Ns_k]*K,
            sigma2=[sigma2.numpy()]*K,
            beta=[1.0]*K,
            eta=eta_np,
            P=[P.numpy()]*Nt,
            max_iter=100
        )
        FA_opt, FD_opt = solver.solve(verbose=False)
        FA_opt = torch.from_numpy(FA_opt).to(cdtype)
        FD_opt = torch.from_numpy(FD_opt).to(cdtype)

        FRF, FBB = HybridPrecodingBFGS(FD_opt, Nrf).solve(max_iter=100, use_B0=True, verbose=False)
        _, papc_ok, info = checkHybridConstraints(FRF, FBB, P, tol=1e-6, verbose=False)
        if not papc_ok:
            FBB = FBB / torch.sqrt(torch.tensor(info["max_papc_ratio"]))
        # ------
        runtime = time.time() - t0

        R, _ = computeWSR(FA_opt, FRF, FBB, H_list, parameters)
        return runtime, R.item()

    def run_Decomp_wrapper(prob_inst):
        H_list = prob_inst["H"]
        K = prob_inst["K"]
        # convert to numpy for AOMO implementation
        H_list_np = [x.numpy() for x in H_list]    
        eta_np = [x.numpy().squeeze() for x in eta]

        t0 = time.time()
        # ------
        solver = DecompBeamformer(
            H=H_list_np,
            Ns_k=[Ns_k]*K,
            sigma2=[sigma2.numpy()]*K,
            beta=[1.0]*K,
            eta=eta_np,
            P=[P.numpy()]*Nt,
            max_iterP1=40,
            max_iterP2=40
        )
        FA_opt, FD_opt = solver.solve(verbose=False)
        FA_opt = torch.from_numpy(FA_opt).to(cdtype)
        FD_opt = torch.from_numpy(FD_opt).to(cdtype)

        FRF, FBB = HybridPrecodingBFGS(FD_opt, Nrf).solve(max_iter=100, use_B0=True, verbose=False)
        _, papc_ok, info = checkHybridConstraints(FRF, FBB, P, tol=1e-6, verbose=False)
        if not papc_ok:
            FBB = FBB / torch.sqrt(torch.tensor(info["max_papc_ratio"]))
        # ------
        runtime = time.time() - t0

        R, _ = computeWSR(FA_opt, FRF, FBB, H_list, parameters)
        return runtime, R.item()

    # =====================================================
    # Benchmark
    # =====================================================
    indices = np.random.choice(len(ProbInsts), NUM_SAMPLES, replace=False)
    print("[Info] Benchmark instances:", indices)

    ALGS = [f"Proposed Unfolded, Iter={I}" for I in UWMMSE_I_LIST] + ["Proposed Model-Based Algo."] + ["Benchmark: AOMO"] + ["Benchmark: Decomp"]
    results = {a: {"rt": [], "sr": []} for a in ALGS}

    for idx in tqdm(indices, desc="Benchmarking"):
        rt, sr = run_classical_WMMSE(ProbInsts[idx])
        results["Proposed Model-Based Algo."]["rt"].append(rt)
        results["Proposed Model-Based Algo."]["sr"].append(sr)

        for I in UWMMSE_I_LIST:
            rt, sr = run_UWMMSE(idx, I)
            results[f"Proposed Unfolded, Iter={I}"]["rt"].append(rt)
            results[f"Proposed Unfolded, Iter={I}"]["sr"].append(sr)

        # Benchmarks
        rt, sr = run_AOMO_wrapper(ProbInsts[idx])
        results["Benchmark: AOMO"]["rt"].append(rt)
        results["Benchmark: AOMO"]["sr"].append(sr)

        rt, sr = run_Decomp_wrapper(ProbInsts[idx])
        results["Benchmark: Decomp"]["rt"].append(rt)
        results["Benchmark: Decomp"]["sr"].append(sr)


    # =====================================================
    # Save results
    # =====================================================
    for name in results:
        rt = np.array(results[name]["rt"])
        sr = np.array(results[name]["sr"])

        np.savez(
            f"{RESULT_DIR}/{name}_WSR_vs_Runtime.npz",
            runtimes=rt,
            sum_rates=sr,
            indices=indices
        )

    print(f"\nData saved to: {RESULT_DIR}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

RESULT_DIR = result_dir("results_WSR_vs_RunTime")

# =====================================================
# style map
# =====================================================
style_map = {
    "Proposed Unfolded, Iter=2": {"marker": "o", "size": 60, "color": "tab:blue"},
    "Proposed Unfolded, Iter=4": {"marker": "o", "size": 60, "color": "tab:orange"},
    "Proposed Unfolded, Iter=6": {"marker": "o", "size": 60, "color": "tab:green"},
    "Proposed Model-Based Algo.": {"marker": "D", "size": 30, "color": "tab:red"},
    "Benchmark: AOMO": {"marker": "x", "size": 90, "color": "black"},
    "Benchmark: Decomp": {"marker": "+", "size": 120, "color": "black"},
}

# =====================================================
# Load all npz files
# =====================================================
plt.figure(figsize=(8, 5))

for file in os.listdir(RESULT_DIR):

    if not file.endswith(".npz"):
        continue

    name = file.replace("_WSR_vs_Runtime.npz", "")

    data = np.load(os.path.join(RESULT_DIR, file))
    rt = data["runtimes"]
    sr = data["sum_rates"]

    style = style_map.get(
        name,
        {"marker": "o", "size": 60, "color": "black"}
    )

    plt.scatter(
        rt, sr,
        s=style["size"],
        marker=style["marker"],
        color=style["color"],
        label=name
    )

# =====================================================
# Plot settings
# =====================================================
plt.xscale("log")
plt.xlabel("Runtime (seconds)")
plt.ylabel("Optimized Sum-rate (bps/Hz)")
plt.title("Runtime vs Optimized Sum-rate")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Summary
print("\n===== Summary =====")
for file in os.listdir(RESULT_DIR):

    if not file.endswith(".npz"):
        continue

    name = file.replace("_WSR_vs_Runtime.npz", "")

    data = np.load(os.path.join(RESULT_DIR, file))
    rt = data["runtimes"]
    sr = data["sum_rates"]

    print(
        f"{name:30s} | "
        f"Runtime {rt.mean():.3f} ± {rt.std():.3f} s | "
        f"WSR {sr.mean():.4f} ± {sr.std():.4f} bps/Hz"
    )

In [ ]:
def print_tikz_table(name, rt, sr):
    print(f"% ===== {name} =====")
    print("table[row sep=crcr]{%")
    for x, y in zip(rt, sr):
        print(f"{x:.6e}\t{y:.6f}\\\\")
    print("};\n")


for file in os.listdir(RESULT_DIR):

    if not file.endswith(".npz"):
        continue

    name = file.replace("_WSR_vs_Runtime.npz", "")

    data = np.load(os.path.join(RESULT_DIR, file))
    rt = data["runtimes"]
    sr = data["sum_rates"]

    print_tikz_table(name, rt, sr)